In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import os
import json
import zipfile
import seaborn as sns

# --- adjustText対応 ---
try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except Exception:
    HAS_ADJUSTTEXT = False

# =============================================================================
# 1. SETTINGS (元コードの設定を維持)
# =============================================================================
base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression"

# ★ 最新の計算結果の日付
NEW_DATE = "20260112" 

OUTPUT_ROOT = f"Final_Thesis_Deliverables_Hybrid_Full_{NEW_DATE}"
OUTPUT_TAG  = f"{NEW_DATE}_v1"
ZIP_NAME    = f"Final_Thesis_Deliverables_{OUTPUT_TAG}.zip"

output_main_dir = OUTPUT_ROOT
os.makedirs(output_main_dir, exist_ok=True)

missing_log_dir = os.path.join(output_main_dir, "Missing_Model_Reports")
os.makedirs(missing_log_dir, exist_ok=True)
missing_records = []

# 解析対象の定義 (元コードと同じ)
target_datasets = ["base", "base_OH", "base_FP", "all", "all_OH", "all_FP"]
target_vars = ["PCEmax", "Jsc", "Voc", "FF"]
regions = ["Inner", "Outer"]
panel_labels = ["A", "B", "C", "D", "E", "F"]

THRES_UP = 1.05
EPSILON = 1e-6
INT_PARAMS = {"ncomp", "C", "degree", "nterms", "k", "min_samples_leaf", "hidden_layer_sizes"}
MAX_LABELS_PER_PANEL = 10
ADJUSTTEXT_LIM = 90
LABEL_OFFSET_FRAC = 0.09
AXIS_MARGIN_FRAC = 0.12

# モデル設定 (元コードと同じ)
model_configs = {
    "PLS":           {"dir": "PLS",           "sub": "20251218_Rebuilt_12Files_PLS_Train_CV10OOF_OOD",            "file": "fixed_20251218_PLS_summary.csv",             "p": ["ncomp"]},
    "svmLinear":     {"dir": "svmLinear",     "sub": "20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD",    "file": "fixed_20251218_SVM_Linear_summary.csv",       "p": ["C"]},
    "gaussprLinear": {"dir": "gaussprLinear", "sub": "20251218_Rebuilt_12Files_GPR_Linear_Train_CV10OOF_OOD",    "file": "fixed_20251218_GPR_Linear_summary.csv",       "p": []},
    "svmRadial":     {"dir": "SVMRadial",     "sub": "20251218_Rebuilt_12Files_SVM_Radial_Train_CV10OOF_OOD",    "file": "fixed_20251218_SVM_Radial_summary.csv",       "p": ["sigma", "C"]},
    "gaussprRadial": {"dir": "gaussPrRadial", "sub": "20251218_Rebuilt_12Files_GPR_Radial_Train_CV10OOF_OOD",    "file": "fixed_20251218_GPR_Radial_summary.csv",       "p": ["sigma"]},
    "gcvEarth":      {"dir": "gcvEarth",      "sub": "20251224_Rebuilt_12Files_gcvEarth_Strict_Final",            "file": "fixed_20251224_gcvEarth_summary.csv",         "p": ["degree"]},
    "ppr":           {"dir": "ppr",           "sub": "20251218_Rebuilt_12Files_PPR_Train_CV10OOF_OOD",            "file": "fixed_20251218_PPR_summary.csv",             "p": ["nterms"]},
    "knn":           {"dir": "knn",           "sub": "20251218_Rebuilt_12Files_kNN_Train_CV10OOF_OOD",            "file": "fixed_20251218_kNN_summary.csv",             "p": ["k"]},
    "Rborist":       {"dir": "Rborist",       "sub": "20251218_Rebuilt_12Files_RboristLikeRF_Train_CV10OOF_OOD", "file": "fixed_20251218_RboristLikeRF_summary.csv",     "p": ["max_features", "min_samples_leaf", "n_estimators", "max_depth"]},
    "monmlp":        {"dir": "monmlp",        "sub": "20251218_Rebuilt_12Files_monmlp_Train_CV10OOF_OOD",         "file": "fixed_20251218_monmlp_summary.csv",           "p": ["alpha", "hidden_layer_sizes"]},
}

ordered_models = list(model_configs.keys())
def disp_model_name(m): return {"ppr": "PPR", "knn": "kNN"}.get(m, m)
ordered_models_disp = [disp_model_name(m) for m in ordered_models]

# ヘルパー関数 (元コードと同じ)
def format_val(v):
    try:
        val = float(v)
        if val == 0: return "0.000"
        if abs(val) >= 0.001: return f"{val:.3f}"
        f = "{:.2e}".format(val); b, e = f.split("e"); return f"${float(b):.2f} \\times 10^{{{int(e)}}}$"
    except: return str(v)

def get_dataset_key(filename):
    fn = str(filename).lower()
    if "base" in fn: return "base_OH" if "oh" in fn else "base_FP" if "fp" in fn else "base"
    if "all" in fn: return "all_OH" if "oh" in fn else "all_FP" if "fp" in fn else "all"
    return "unknown"

import re
import json
import pandas as pd

def parse_param(p_str, p_name):
    if pd.isna(p_str) or p_str == "" or p_str == "N/A":
        return "N/A"
    
    extracted = "N/A"
    p_str_clean = str(p_str).replace("'", '"')
    
    # 1. JSON形式としてのパースを試行
    if p_str_clean.startswith("{"):
        try:
            d = json.loads(p_str_clean)
            if p_name in d:
                extracted = str(d[p_name])
        except:
            pass

    # 2. 正規表現によるテキストパース (JSONパース失敗時、または値がN/Aの場合)
    if extracted == "N/A":
        # カッコ () 内のカンマを含めて抽出するか、単純な数値/文字列を抽出する正規表現
        # 構造: キー名 + 区切り( = or : ) + 値(カッコ内 or 数値 or 文字列)
        pattern = rf'["\']?{p_name}["\']?\s*[:=]\s*(?P<val>\([^)]+\)|[0-9\.eE\-\+]+|[^,\s}}]+)'
        m = re.search(pattern, str(p_str))
        if m:
            extracted = m.group('val')

    # 3. 抽出成功時のクリーンアップ処理
    if extracted != "N/A":
        s = extracted.strip().strip('"').strip("'")
        
        # null / None の変換
        if s.lower() in ["null", "none", "nan"]:
            return "None"
        
        # monmlp 対策: (128, 0.001) のようなカッコ構造の整形
        if "(" in s:
            s = s.replace("(", "").replace(")", "") # カッコを除去
            if "," in s:
                s = s.split(",")[0].strip() # 最初の要素(ユニット数など)のみ取得
        
        # 数値変換とフォーマット
        try:
            # 既にカッコを除去した純粋な数値文字列を処理
            fv = float(s)
            # 整数として扱うべきパラメータリスト (環境に合わせて調整)
            INT_PARAMS = ["ncomp", "n_estimators", "max_depth", "min_samples_leaf", "k", "nk", "nterms", "degree"]
            
            if p_name in INT_PARAMS:
                return str(int(round(fv)))
            return format_val(fv) # 小数点以下の整形関数（既存）
        except:
            # 数値変換できない場合はそのまま（"sqrt" など）を返す
            return s
            
    return "N/A"

def annotate_with_adjusttext_far(ax, xs, ys, labels, fontsize=11):
    if len(xs) == 0: return
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    dx, dy = (x1 - x0) * LABEL_OFFSET_FRAC, (y1 - y0) * LABEL_OFFSET_FRAC
    ann_list = []
    for x, y, lab in zip(xs, ys, labels):
        ann = ax.annotate(lab, xy=(x, y), xytext=(x + dx, y + dy), textcoords="data",
                          fontsize=fontsize, fontweight="bold", ha="left", va="bottom",
                          bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.75),
                          arrowprops=dict(arrowstyle="-", color="gray", lw=1), zorder=3)
        ann_list.append(ann)
    if HAS_ADJUSTTEXT:
        adjust_text(ann_list, x=xs, y=ys, ax=ax, expand_points=(3.2, 3.2), expand_text=(1.8, 2.2),
                    force_points=0.25, force_text=1.10, only_move={'points': 'xy', 'text': 'xy'}, lim=ADJUSTTEXT_LIM)

# =============================================================================
# 2. DATA COLLECTION (ここを新規「ブロック1」詳細ログ付きに差し替え)
# =============================================================================
print(f"\n" + "="*80)
print(f"Step 1: DATA COLLECTION (Hybrid Mode)")
print(f"  - Target Date for m_all: {NEW_DATE}")
print(f"  - Base Directory: {base_path}")
print("="*80)

all_rows = []
for mod, cfg in model_configs.items():
    print(f"\n[Model: {mod}]")
    old_p = os.path.join(base_path, cfg["dir"], cfg["sub"], cfg["file"])
    new_sub = re.sub(r"2025\d{4}", NEW_DATE, cfg["sub"])
    new_f = re.sub(r"2025\d{4}", NEW_DATE, cfg["file"])
    new_p = os.path.join(base_path, cfg["dir"], new_sub, new_f)
    
    df_o = pd.read_csv(old_p) if os.path.exists(old_p) else None
    df_n = pd.read_csv(new_p) if os.path.exists(new_p) else None
    
    if df_o is not None:
        df_old_f = df_o[~df_o['File'].str.startswith('m_all')].copy()
        print(f"  - OLD SOURCE: {old_p}\n    -> Loaded {len(df_old_f)} rows (n_base, n_all, m_base)")
    else: df_old_f = pd.DataFrame(); print(f"  - OLD SOURCE: Not found.")

    if df_n is not None:
        df_new_f = df_n[df_n['File'].str.startswith('m_all')].copy()
        print(f"  - NEW SOURCE: {new_p}\n    -> Loaded {len(df_new_f)} rows (m_all: Leakage-fixed)")
    else:
        df_new_f = df_o[df_o['File'].str.startswith('m_all')].copy() if df_o is not None else pd.DataFrame()
        print(f"  - WARNING: New m_all not found. Falling back to old results.")

    df_comb = pd.concat([df_old_f, df_new_f], axis=0).drop_duplicates()
    for _, row in df_comb.iterrows():
        all_rows.append({
            "Model": mod, "Target": row["Target"], "Dataset": get_dataset_key(row["File"]),
            "Imp": str(row["File"])[0], "Inner_R2": row["CV10_R2"], "Inner_RMSE": row["CV10_RMSE"],
            "Outer_R2": row["OOD_R2"], "Outer_RMSE": row["OOD_RMSE"], "Best_Params": row["Best_Params"]
        })

master_df = pd.DataFrame(all_rows)
print(f"\nStep 1 Complete: Total {len(master_df)} records.")

# =============================================================================
# 3. CONTINUE TO元コード Step 2 (ここから下は元コードの175行目以降をそのままペースト)
# =============================================================================
print("Step 2: Generating Type 1 Summary files...")
# ... [以下、元コードのStep 2からZIP処理までをコピー] ...


Step 1: DATA COLLECTION (Hybrid Mode)
  - Target Date for m_all: 20260112
  - Base Directory: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression

[Model: PLS]
  - OLD SOURCE: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/PLS/20251218_Rebuilt_12Files_PLS_Train_CV10OOF_OOD/fixed_20251218_PLS_summary.csv
    -> Loaded 36 rows (n_base, n_all, m_base)
  - NEW SOURCE: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/PLS/20260112_Rebuilt_12Files_PLS_Train_CV10OOF_OOD/fixed_20260112_PLS_summary.csv
    -> Loaded 12 rows (m_all: Leakage-fixed)

[Model: svmLinear]
  - OLD SOURCE: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/svmLinear/202512

In [18]:
# --- 4. Step2: Type1 Summary (16 files) --------------------------------------
print("Step 2: Generating Type 1 Summary files...")
type1_dir = os.path.join(output_main_dir, "Summary_Files_16")
os.makedirs(type1_dir, exist_ok=True)

for reg in regions:
    r_label = "Inner" if reg == "Inner" else "Outer"
    for imp in ["n", "m"]:
        for tgt in target_vars:
            rows = []
            for mod in ordered_models:
                sub = master_df[(master_df["Model"] == mod) & (master_df["Target"] == tgt) & (master_df["Imp"] == imp)]
                if sub.empty:
                    continue

                score_row = {"Model": disp_model_name(mod)}
                for ds in target_datasets:
                    m = sub[sub["Dataset"] == ds]
                    if not m.empty:
                        score_row[ds] = f"{format_val(m.iloc[0][f'{r_label}_R2'])} ({format_val(m.iloc[0][f'{r_label}_RMSE'])})"
                    else:
                        score_row[ds] = "N/A"
                rows.append(score_row)

                for pn in model_configs[mod]["p"]:
                    p_row = {"Model": f"({pn})"}
                    for ds in target_datasets:
                        m = sub[sub["Dataset"] == ds]
                        p_row[ds] = parse_param(m.iloc[0]["Best_Params"], pn) if not m.empty else "N/A"
                    rows.append(p_row)

            out_path = os.path.join(type1_dir, f"{OUTPUT_TAG}_Summary_{reg}_{imp}_{tgt}.csv")
            pd.DataFrame(rows).to_csv(out_path, index=False)

# --- 5. Step3: Type2 Detailed Ratio Matrices ---------------------------------
print("Step 3: Generating Type 2 Detailed Matrices (with 2 judges)...")
type2_dir = os.path.join(output_main_dir, "Detailed_Matrices")
os.makedirs(type2_dir, exist_ok=True)
def annotate_with_adjusttext_far(ax, xs, ys, labels, *,
                                fontsize=11, fontweight="bold",
                                lim=ADJUSTTEXT_LIM,
                                offset_frac=LABEL_OFFSET_FRAC):
    """
    矢印が必ず点(x,y)を指す adjustText 対応版。
    引数: fontweight, lim, offset_frac をすべて受け取れるように拡張。
    """
    if len(xs) == 0:
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    dx = (x1 - x0) * offset_frac
    dy = (y1 - y0) * offset_frac

    ann_list = []
    for x, y, lab in zip(xs, ys, labels):
        ann = ax.annotate(
            lab,
            xy=(x, y),                  # 矢印の先端を点に固定
            xytext=(x + dx, y + dy),    # テキストの初期位置
            textcoords="data",
            fontsize=fontsize,
            fontweight=fontweight,
            ha="left", va="bottom",
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.75),
            arrowprops=dict(arrowstyle="-", color="gray", lw=1),
            zorder=3
        )
        ann_list.append(ann)

    if HAS_ADJUSTTEXT:
        adjust_text(
            ann_list,
            x=xs, y=ys, ax=ax,
            expand_points=(3.2, 3.2),
            expand_text=(1.8, 2.2),
            force_points=0.25,
            force_text=1.10,
            only_move={'points': 'xy', 'text': 'xy'},
            lim=lim
        )

def build_type2_matrix(metric_kind):
    final_rows = []
    for mod in ordered_models:
        mod_disp = disp_model_name(mod)

        row_v  = {"Model": mod_disp}
        row_j5 = {"Model": f"({mod_disp}_Judge_5pc)"}
        row_jb = {"Model": f"({mod_disp}_Judge_Binary)"}

        for reg in regions:
            prefix = "CV" if reg == "Inner" else "OOD"
            col_metric = f"{reg}_{metric_kind}"

            for tgt in target_vars:
                for ds in target_datasets:
                    col = f"{metric_kind}_Ratio_{prefix}_{tgt}_{ds}"

                    sn = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="n")]
                    sm = master_df[(master_df["Model"]==mod) & (master_df["Target"]==tgt) & (master_df["Dataset"]==ds) & (master_df["Imp"]=="m")]

                    val, j5, jb = "N/A", 0, -1
                    if not sn.empty and not sm.empty:
                        nv = sn.iloc[0][col_metric]
                        mv = sm.iloc[0][col_metric]

                        if pd.isna(nv) or pd.isna(mv):
                            val, j5, jb = "N/A", 0, -1
                        elif abs(float(nv)) > EPSILON:
                            ratio = float(mv) / float(nv)
                            val = f"{ratio:.3f}"

                            if metric_kind == "R2":
                                j5 = 1 if ratio >= THRES_UP else (-1 if ratio <= 1/THRES_UP else 0)
                                jb = 1 if ratio >= 1.0 else -1
                            else:
                                j5 = 1 if ratio <= 0.95 else (-1 if ratio >= 1.05 else 0)
                                jb = 1 if ratio <= 1.0 else -1
                        elif abs(float(nv)) <= EPSILON and abs(float(mv)) > EPSILON:
                            val, j5, jb = "Inf", 1, 1

                    row_v[col], row_j5[col], row_jb[col] = val, j5, jb

        final_rows.extend([row_v, row_j5, row_jb])
    return pd.DataFrame(final_rows)

for m_kind in ["R2", "RMSE"]:
    df2 = build_type2_matrix(m_kind)
    out_path = os.path.join(type2_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_{m_kind}_Combined_Judges.csv")
    df2.to_csv(out_path, index=False)

# --- 5.5 Step3.5: Heatmaps (your seaborn logic unchanged) ---------------------
print("Step 3.5: Generating Heatmaps (your seaborn version, unchanged logic)...")
import seaborn as sns

heat_dir = os.path.join(output_main_dir, "Heatmaps_Ratio_Styled")
os.makedirs(heat_dir, exist_ok=True)

def create_styled_heatmap(file_path, title, output_img, is_rmse=False):
    df = pd.read_csv(file_path)

    model_rows = df[~df['Model'].str.startswith('(')].copy()
    judge_rows = df[df['Model'].str.contains('Judge_5pc')].copy()

    model_rows.set_index('Model', inplace=True)
    judge_rows.set_index('Model', inplace=True)

    judge_rows.index = judge_rows.index.str.extract(r'\((.*)_Judge_5pc\)')[0]

    val_df = model_rows.apply(pd.to_numeric, errors='coerce')
    judge_df = judge_rows.apply(pd.to_numeric, errors='coerce')

    pref_order = ["PLS", "svmLinear", "gaussprLinear", "svmRadial", "gaussprRadial",
                  "gcvEarth", "PPR", "kNN", "Rborist", "monmlp"]
    available_models = [m for m in pref_order if m in val_df.index]
    val_df = val_df.loc[available_models]
    judge_df = judge_df.loc[available_models]

    plt.figure(figsize=(20, 8))
    from matplotlib.colors import LinearSegmentedColormap
    colors = ["#ff9999", "#ffffff", "#99ff99"]
    cmap = LinearSegmentedColormap.from_list("judge_cmap", colors, N=3)

    sns.heatmap(
        judge_df,
        annot=val_df, fmt=".3f",
        cmap=cmap, center=0,
        linewidths=.5,
        cbar_kws={'label': 'Judgment (-1: Loss, 0: Neutral, 1: Win)'},
        annot_kws={"size": 10}
    )

    plt.title(title, fontsize=16, pad=20)
    plt.xlabel("Target Variable & Descriptor Dataset", fontsize=12)
    plt.ylabel("ML Model", fontsize=12)
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(output_img, dpi=300)
    plt.close()

def save_filtered_matrix(src_csv, dst_csv, prefix_keep):
    df = pd.read_csv(src_csv)
    keep_cols = ["Model"] + [c for c in df.columns if c.startswith(prefix_keep)]
    df2 = df[keep_cols].copy()
    df2.to_csv(dst_csv, index=False)

# Use the generated Type2 outputs for heatmaps
src_r2   = os.path.join(type2_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_R2_Combined_Judges.csv")
src_rmse = os.path.join(type2_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_RMSE_Combined_Judges.csv")

cv_r2_csv    = os.path.join(heat_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_R2_onlyCV.csv")
ood_r2_csv   = os.path.join(heat_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_R2_onlyOOD.csv")
cv_rmse_csv  = os.path.join(heat_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_RMSE_onlyCV.csv")
ood_rmse_csv = os.path.join(heat_dir, f"{OUTPUT_TAG}_Detailed_Ratio_Matrix_RMSE_onlyOOD.csv")

save_filtered_matrix(src_r2,   cv_r2_csv,    "R2_Ratio_CV_")
save_filtered_matrix(src_r2,   ood_r2_csv,   "R2_Ratio_OOD_")
save_filtered_matrix(src_rmse, cv_rmse_csv,  "RMSE_Ratio_CV_")
save_filtered_matrix(src_rmse, ood_rmse_csv, "RMSE_Ratio_OOD_")

create_styled_heatmap(
    cv_r2_csv,
    "MICE Imputation Effect: R2 Ratio (m_R2 / n_R2) - CV Results",
    os.path.join(heat_dir, f"{OUTPUT_TAG}_Heatmap_CV_R2_Ratio.png"),
    is_rmse=False
)
create_styled_heatmap(
    cv_rmse_csv,
    "MICE Imputation Effect: RMSE Ratio (m_RMSE / n_RMSE) - CV Results",
    os.path.join(heat_dir, f"{OUTPUT_TAG}_Heatmap_CV_RMSE_Ratio.png"),
    is_rmse=True
)
create_styled_heatmap(
    ood_r2_csv,
    "MICE Imputation Effect: R2 Ratio (m_R2 / n_R2) - OOD Results",
    os.path.join(heat_dir, f"{OUTPUT_TAG}_Heatmap_OOD_R2_Ratio.png"),
    is_rmse=False
)
create_styled_heatmap(
    ood_rmse_csv,
    "MICE Imputation Effect: RMSE Ratio (m_RMSE / n_RMSE) - OOD Results",
    os.path.join(heat_dir, f"{OUTPUT_TAG}_Heatmap_OOD_RMSE_Ratio.png"),
    is_rmse=True
)

print("[DONE] Heatmaps created.")

# --- 6. Step4: Type3 Statistics ----------------------------------------------
print("Step 4: Generating Type 3 Statistics...")
type3_dir = os.path.join(output_main_dir, "Statistics_Summaries")
os.makedirs(type3_dir, exist_ok=True)

compare_list = []
for (mod, tgt, ds), group in master_df.groupby(["Model", "Target", "Dataset"]):
    rn = group[group["Imp"] == "n"]
    rm = group[group["Imp"] == "m"]
    if not rn.empty and not rm.empty:
        res = {"Model": disp_model_name(mod), "Target": tgt, "Dataset": ds}
        for reg in regions:
            for mtr in ["R2", "RMSE"]:
                nv = rn.iloc[0][f"{reg}_{mtr}"]
                mv = rm.iloc[0][f"{reg}_{mtr}"]
                res[f"n_{mtr}_{reg}"] = nv
                res[f"m_{mtr}_{reg}"] = mv
                if mtr == "R2":
                    res[f"Win_{mtr}_{reg}"] = 1 if float(mv) >= float(nv) - EPSILON else 0
                else:
                    res[f"Win_{mtr}_{reg}"] = 1 if float(mv) <= float(nv) + EPSILON else 0
        compare_list.append(res)

comp_df = pd.DataFrame(compare_list)

for axis in ["Target", "Model", "Dataset"]:
    summary = comp_df.groupby(axis).agg({
        **{f"{i}_{mtr}_{reg}": "mean" for i in ["n", "m"] for mtr in ["R2", "RMSE"] for reg in regions},
        **{f"Win_{mtr}_{reg}": "sum" for mtr in ["R2", "RMSE"] for reg in regions},
        "Model": "count"
    }).rename(columns={"Model": "Total_Tests"})

    for c in summary.columns:
        if c != "Total_Tests":
            summary[c] = summary[c].apply(format_val)

    if axis == "Model":
        summary = summary.reindex(ordered_models_disp)

    out_path = os.path.join(type3_dir, f"{OUTPUT_TAG}_Summary_Statistics_By_{axis}.csv")
    summary.to_csv(out_path)

# --- 7. Step5: Panel Scatter Plots -------------------------------------------
print("Step 5: Generating 2x3 Panel Plots (adjustText, labels behind points, avoid-point)...")
type4_dir = os.path.join(output_main_dir, "Scatter_Plots_Panels")
os.makedirs(type4_dir, exist_ok=True)

for tgt in target_vars:
    for split in regions:
        for m_kind in ["R2", "RMSE"]:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            axes = axes.flatten()

            for i, ds in enumerate(target_datasets):
                sub_n = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "n") & (master_df["Dataset"] == ds)]
                sub_m = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "m") & (master_df["Dataset"] == ds)]

                # --- Missing model diagnostics (per panel) -----------------------------------
                def safe_get_val(dfsub, col):
                    if dfsub.empty:
                        return None
                    v = dfsub.iloc[0].get(col, None)
                    if pd.isna(v):
                        return None
                    try:
                        return float(v)
                    except:
                        return None

                # sub_n, sub_m は既に作っている前提（Target, Imp, Dataset で絞ったもの）
                expected_models = ordered_models[:]  # internal names

                missing_models = []
                for mod in expected_models:
                    rn = sub_n[sub_n["Model"] == mod]
                    rm = sub_m[sub_m["Model"] == mod]

                    # “プロットに出る条件”＝ nとm両方が揃い、かつ split×m_kind が数値
                    vn = safe_get_val(rn, f"{split}_{m_kind}")
                    vm = safe_get_val(rm, f"{split}_{m_kind}")

                    if (vn is None) or (vm is None):
                        missing_models.append(mod)

                        # n/m の R2/RMSE を inner/outer 両方まとめて記録
                        rec = {
                            "Target": tgt,
                            "Dataset": ds,
                            "Region(split)": split,     # Inner/Outer
                            "Metric(plotted)": m_kind,  # R2/RMSE
                            "Missing_Model": disp_model_name(mod),
                            # n values
                            "n_Inner_R2":  safe_get_val(rn, "Inner_R2"),
                            "n_Inner_RMSE": safe_get_val(rn, "Inner_RMSE"),
                            "n_Outer_R2":  safe_get_val(rn, "Outer_R2"),
                            "n_Outer_RMSE": safe_get_val(rn, "Outer_RMSE"),
                            # m values
                            "m_Inner_R2":  safe_get_val(rm, "Inner_R2"),
                            "m_Inner_RMSE": safe_get_val(rm, "Inner_RMSE"),
                            "m_Outer_R2":  safe_get_val(rm, "Outer_R2"),
                            "m_Outer_RMSE": safe_get_val(rm, "Outer_RMSE"),
                            # raw presence flags
                            "has_n_row": (not rn.empty),
                            "has_m_row": (not rm.empty),
                            "has_n_metric_value": (vn is not None),
                            "has_m_metric_value": (vm is not None),
                        }
                        missing_records.append(rec)

                # コンソール表示（欠けがある時だけ）
                if len(missing_models) > 0:
                    print("\n[WARN] Missing models in plot panel:")
                    print(f"  - Target={tgt} | Dataset={ds} | split={split} | metric={m_kind}")
                    print("  - Missing:", ", ".join(disp_model_name(m) for m in missing_models))

                ax = axes[i]

#                 sub_n = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "n") & (master_df["Dataset"] == ds)]
#                 sub_m = master_df[(master_df["Target"] == tgt) & (master_df["Imp"] == "m") & (master_df["Dataset"] == ds)]

                x_v, y_v, names = [], [], []
                for mod in ordered_models:
                    rn = sub_n[sub_n["Model"] == mod]
                    rm = sub_m[sub_m["Model"] == mod]
                    if not rn.empty and not rm.empty:
                        vn = rn.iloc[0][f"{split}_{m_kind}"]
                        vm = rm.iloc[0][f"{split}_{m_kind}"]
                        if not (pd.isna(vn) or pd.isna(vm)):
                            x_v.append(float(vn))
                            y_v.append(float(vm))
                            names.append(disp_model_name(mod))

                ax.text(0.05, 0.95, panel_labels[i], transform=ax.transAxes,
                        fontsize=28, fontweight="bold", va="top")
                ax.set_title(ds, fontsize=16, fontweight="bold")
                ax.set_xlabel(f"{m_kind} (n)", fontsize=14, fontweight="bold")
                ax.set_ylabel(f"{m_kind} (m)", fontsize=14, fontweight="bold")

                if not x_v:
                    ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
                    continue

                # Points on top (so labels don't visually cover points)
                ax.scatter(x_v, y_v, s=100, edgecolors="black", alpha=0.85, zorder=10)

                # Limits
                if m_kind == "R2":
                    lims = [0.0, 1.0]
                else:
                    allv = x_v + y_v
                    mn = min(allv)
                    mx = max(allv)
                    if mx == mn:
                        lims = [mn - 0.5, mx + 0.5]
                    else:
                        pad = 0.1 * (mx - mn)
                        lims = [mn - pad, mx + pad]

                ax.plot(lims, lims, linestyle="--", alpha=0.6, zorder=1)

                # Give extra margin for labels
                pad2 = AXIS_MARGIN_FRAC * (lims[1] - lims[0])
                ax.set_xlim(lims[0] - pad2, lims[1] + pad2)
                ax.set_ylim(lims[0] - pad2, lims[1] + pad2)
                ax.set_aspect("equal", adjustable="box")

                # Label only top-K improvements (speed/readability)
#                 x_lab, y_lab, name_lab = pick_topK_labels(x_v, y_v, names, m_kind, K=MAX_LABELS_PER_PANEL)

#                 annotate_with_adjusttext_far(
#                     ax, x_lab, y_lab, name_lab,
#                     fontsize=11, fontweight="bold",
#                     lim=ADJUSTTEXT_LIM,
#                     offset_frac=LABEL_OFFSET_FRAC
#                 )
                annotate_with_adjusttext_far(
                    ax, x_v, y_v, names,
                    fontsize=11, fontweight="bold",
                    lim=ADJUSTTEXT_LIM,
                    offset_frac=LABEL_OFFSET_FRAC
                )

            plt.tight_layout()
            out_path = os.path.join(type4_dir, f"{OUTPUT_TAG}_Panel_{tgt}_{split}_{m_kind}.png")
            plt.savefig(out_path, dpi=300)
            plt.close()

# --- 8. Step6: ZIP ------------------------------------------------------------
print("Step 6: Zipping all results...")

if len(missing_records) > 0:
    miss_df = pd.DataFrame(missing_records)
    out_miss = os.path.join(missing_log_dir, f"{OUTPUT_TAG}_MissingModels_AllPanels.csv")
    miss_df.to_csv(out_miss, index=False)
    print(f"\n[SAVED] Missing-model report CSV -> {out_miss}")
else:
    print("\n[INFO] No missing models detected in any panel.")

zip_path = os.path.join(base_path, ZIP_NAME)


with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(output_main_dir):
        for file in files:
            full = os.path.join(root, file)
            rel = os.path.relpath(full, output_main_dir)
            zipf.write(full, rel)

print(f"\n=== ALL SUCCESS ===\nResults saved to folder: {output_main_dir}\nZip created at: {zip_path}")


Step 2: Generating Type 1 Summary files...
Step 3: Generating Type 2 Detailed Matrices (with 2 judges)...
Step 3.5: Generating Heatmaps (your seaborn version, unchanged logic)...
[DONE] Heatmaps created.
Step 4: Generating Type 3 Statistics...
Step 5: Generating 2x3 Panel Plots (adjustText, labels behind points, avoid-point)...
8 [-0.94929263 -0.6995507 ]
9 [-0.26445201 -0.82564134]
8 [-0.2434623  -0.57287402]
9 [-0.80525212 -0.86179904]
8 [-0.34444066  0.11928659]
9 [0.90296712 0.61240619]
8 [-0.98086484 -0.86112947]
9 [-0.80491323 -0.88002077]
8 [0.75703044 0.81477752]
9 [0.30570459 0.72462781]
8 [-0.5926065  -0.40554308]
9 [-0.57753105 -0.52481324]
8 [-0.8110654  -0.11407109]
9 [-0.35060913 -0.93991996]
8 [-0.02397561 -0.563769  ]
9 [ 0.35148391 -0.20245406]
8 [-0.71612295 -0.79053622]
9 [-0.1331872   0.91863926]
8 [-0.4505001  -0.75063316]
9 [0.67903948 0.84296518]
Step 6: Zipping all results...

[INFO] No missing models detected in any panel.

=== ALL SUCCESS ===
Results saved to 